## Visualize in Image (BEV) space

In [ ]:
import cv2
import numpy as np
from pathlib import Path


def load_bev_predictions(pred_path, min_conf):
    predictions = []
    
    with open(pred_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 6:  # class, x, y, w, h, rotation, confidence
                    class_id = int(parts[0])
                    x_center = float(parts[1])
                    y_center = float(parts[2])
                    width = float(parts[3])
                    height = float(parts[4])
                    rotation = float(parts[5])
                    confidence = float(parts[6]) if len(parts) > 6 else 1.0
                    
                    if confidence >= min_conf:
                        predictions.append([class_id, x_center, y_center, width, height, rotation, confidence])
    
    return predictions

def load_bev_groundtruth(gt_path, image_shape=(640, 640)):
    groundtruth = []

    with open(gt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 9:
                    class_id = int(parts[0])
                    # [x1, y1, x2, y2, x3, y3, x4, y4]
                    corners_8 = np.array([float(parts[i]) for i in range(1, 9)])

                    height, width = image_shape[:2]
                    corners_8[0::2] *= width
                    corners_8[1::2] *= height

                    cx, cy, w, h, rotation = xyxyxyxy2xywhr(corners_8)

                    groundtruth.append([class_id, cx, cy, w, h, rotation])
    
    return groundtruth

def xyxyxyxy2xywhr(corners_8):
    # [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
    points = corners_8.reshape(4, 2).astype(np.float32)

    (cx, cy), (w, h), angle = cv2.minAreaRect(points)

    rotation = angle / 180 * np.pi

    return [cx, cy, w, h, rotation]

def xywhr_to_rotated_corners(x_center, y_center, width, height, rotation):

    cos_rot, sin_rot = np.cos(rotation), np.sin(rotation)
    
    # Define corners relative to center (before rotation)
    corners = np.array([
        [-width/2, -height/2],  # Bottom-left
        [width/2, -height/2],   # Bottom-right
        [width/2, height/2],    # Top-right
        [-width/2, height/2]    # Top-left
    ])
    
    # Apply rotation
    rot_matrix = np.array([[cos_rot, -sin_rot], [sin_rot, cos_rot]])
    rotated_corners = corners @ rot_matrix.T
    
    # Translate to final position
    final_corners = rotated_corners + [x_center, y_center]
    
    return final_corners.astype(np.int32)

def draw_rotated_box(image, corners, class_id, confidence=None, class_names=None, colors=None, is_gt=False):

    if colors is None:
        colors = {1: (0, 255, 0), # Car: Green
                  2: (235, 181, 35), # Pedestrian: Blue
                  3: (0, 255, 255)} # Cyclist: Yellow
    
    if class_names is None:
        class_names = {1: "Car", 2: "Ped.", 3: "Cyc."}

    if is_gt:
        color = (0, 0, 255)
        thickness = 1
    else:
        color = colors.get(class_id, (255, 255, 255))
        thickness = 1
    
    cv2.polylines(image, [corners], isClosed=True, color=color, thickness=thickness)
    
    if is_gt:
        label = f"{class_names.get(class_id, f'Class{class_id}')}"
    else:
        # Add label
        label = f"{class_names.get(class_id, f'Class{class_id}')} {confidence:.3f}"
    
    # Find top-left corner for label placement
    top_left = corners[np.argmin(corners[:, 1] + corners[:, 0])]
    label_pos = (top_left[0], top_left[1] - 10)

    if label:
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.4
        font_thickness = 1

        cv2.putText(image, label, label_pos, font, font_scale, color, font_thickness)

def visualize_bev(image_path, pred_path, gt_path=None, save=False, output_path=None, show_image=True, min_conf=0.0):
    
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"Failed to load image: {image_path}")
        return None    
    
    # Load and draw predictions
    if Path(pred_path).exists(): 
        predictions = load_bev_predictions(pred_path, min_conf)
        print(f"\nLoaded {len(predictions)} predictions from {Path(pred_path).name}")

        for pred in predictions:
            class_id, x_center, y_center, width, height, rotation, confidence = pred
            corners = xywhr_to_rotated_corners(x_center, y_center, width, height, rotation)
            draw_rotated_box(image, corners, class_id, confidence, is_gt=False)
            print(f"Class: {class_id}, Center: ({x_center:.1f}, {y_center:.1f}), "
              f"Size: {width:.1f}x{height:.1f}, Rotation: {rotation:.3f}, Conf: {confidence:.3f}")
    else:
        print(f"Predictions not found: {pred_path}")
    
    if gt_path and Path(gt_path).exists():
        groundtruths = load_bev_groundtruth(gt_path) 
        for gt in groundtruths:
            class_id, x_center, y_center, width, height, rotation = gt
            corners = xywhr_to_rotated_corners(x_center, y_center, width, height, rotation)
            draw_rotated_box(image, corners, class_id, is_gt=True)
    elif gt_path:
        print(f"GT file not found: {gt_path}")

    # Save result if output path provided
    if save:
        cv2.imwrite(output_path, image)

    # Show image window (optional)
    if show_image:
        cv2.imshow('BEV Predictions vs. GT', image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    
    return image

if __name__ == "__main__":
    # Define paths
    base_dir = Path("/home/heizung1/view-of-delft-dataset/vod/label_transformation")
    # Update this with actual BEV image path
    image_path = Path("/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/cfg/datasets/kitti/images/val/bev_val_000006.png")
    
    # [-pi/4...3pi/4]
    pred_path = base_dir / "predictions/all_bev_preds_minAreaRect()/val_trt_fp32/labels/bev_val_000006.txt"
    # [0...pi/2]
    #pred_path = base_dir / "predictions/all_bev_preds_regularized/val_trt_fp32_rgd/labels/bev_val_000006.txt"

    gt_path = base_dir / "kitti_gt_annos_2/all_bev_gt_annos_2/bev_val_000006.txt"

    output_path = base_dir / "images/BEV Predictions vs. GT_test.png"
    
    if image_path.exists():
        print(f"Using image: {image_path}")
        print(f"Using labels: {pred_path}")
        
        # Visualize predictions (saves image, no display)
        result_image = visualize_bev(
            image_path=image_path,
            pred_path=pred_path,
            gt_path=gt_path,
            save=False,
            output_path=output_path,
            min_conf=0.01,
            show_image=True
        )
    else:
        print("BEV image not found. Please update the image_path.")
        # Still show label info
        if pred_path.exists():
            predictions = load_bev_predictions(pred_path)
            for i, pred in enumerate(predictions):
                print(f"  {i+1}: Class {pred[0]}, Center ({pred[1]:.1f}, {pred[2]:.1f}), "
                      f"Size {pred[3]:.1f}x{pred[4]:.1f}, Rot {pred[5]:.3f}, Conf {pred[6]:.3f}")

## Visualize in LiDAR frame

In [ ]:
import numpy as np
from pathlib import Path
import pickle
import matplotlib.cm as cm

from vod.visualization import open3d_vis_utils as Visualizer

CLASS_MAP = {"Car": 1, "Pedestrian": 2, "Cyclist": 3}

def load_lidar_gt_labels(label_path):
    boxes = []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            obj_type = parts[0]
            h, w, l = map(float, parts[8:11])
            x, y, z = map(float, parts[11:14])
            rot_z = float(parts[14])
            class_idx = CLASS_MAP.get(obj_type, 0)
            boxes.append([x, y, z, l, w, h, rot_z, class_idx])
    return np.array(boxes, dtype=np.float32)

def load_lidar_pred_labels(label_path, min_confidence=0.01):
    boxes, scores, labels = [], [], []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            obj_type = parts[0]
            h, w, l = map(float, parts[8:11])
            x, y, z = map(float, parts[11:14])
            rot_z = float(parts[14])
            confidence = float(parts[15])
            class_idx = CLASS_MAP.get(obj_type, 0)
            
            if confidence >= min_confidence:
                boxes.append([x, y, z, l, w, h, rot_z, class_idx])
                scores.append(confidence)
                labels.append(class_idx)
    return np.array(boxes, dtype=np.float32), np.array(scores), np.array(labels)

def load_pc_scene(pkl_path, lidar_idx=None):
    with open(pkl_path, 'rb') as f:
            val_data = pickle.load(f)

    for scene_data in val_data:
        if scene_data['point_cloud']['lidar_idx'] == lidar_idx:
            return scene_data['points']

if __name__ == "__main__":
    base_dir = Path("/home/heizung1/view-of-delft-dataset/vod/label_transformation")
    gt_path = base_dir / "kitti_gt_annos_2/gt_bev_to_lidar_labels_2/000006.txt"
    # [pi/4...3pi/4]
    pred_path = base_dir / "predictions/all_bev_preds_minAreaRect()/pred_bev_to_lidar_fp32/000006.txt"
    # [0...pi/2]
    #pred_path = base_dir / "predictions/all_bev_preds_regularized/pred_bev_to_lidar_fp32_rgd/000006.txt"

    pkl_path = "validation_pickle/kitti_val_dataset.pkl"
    output_path = base_dir / "images/3d_predictions_result.png"

    lidar_idx = gt_path.stem
    points = load_pc_scene(pkl_path, lidar_idx)
    gt_boxes = load_lidar_gt_labels(gt_path)
    pred_boxes, pred_scores, pred_labels = load_lidar_pred_labels(pred_path, min_confidence=0.01)

    distances = np.linalg.norm(points[:, :3], axis=1)
    normalized_distances = (distances - np.min(distances)) / (np.max(distances) - np.min(distances))
    colormap_options = ['hot', 'autumn']
    cmap = cm.get_cmap('autumn')
    point_colors = cmap(normalized_distances)[:, :3]

    Visualizer.draw_scenes(
        points=points, 
        gt_boxes=gt_boxes, 
        ref_boxes=pred_boxes, 
        ref_labels=pred_labels,
        ref_scores=pred_scores,
        point_colors=point_colors,
        draw_origin=True, 
        save_image=False, 
        output_path=output_path,
        draw_obj_heading=True)

# Open3D controls
"""
-- Mouse view control --
  Left button + drag         : Rotate.
  Ctrl + left button + drag  : Translate.
  Wheel button + drag        : Translate.
  Shift + left button + drag : Roll.
  Wheel                      : Zoom in/out.

-- Keyboard view control --
  [/]          : Increase/decrease field of view.
  R            : Reset view point.
  Ctrl/Cmd + C : Copy current view status into the clipboard.
  Ctrl/Cmd + V : Paste view status from clipboard.

-- General control --
  Q, Esc       : Exit window.
  H            : Print help message.
  P, PrtScn    : Take a screen capture.
  D            : Take a depth capture.
  O            : Take a capture of current rendering settings.
"""


## Visualize in Camera frame

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import pickle

CLASS_MAP = {"Car": 1, "Pedestrian": 2, "Cyclist": 3}

box_colormap = [
    [1, 1, 1],  # not assigned - White
    [0, 1, 0],  # Car - Green
    [1, 0, 1],  # Pedestrian - Violet
    [1, 1, 0],  # Cyclist - Yellow
]

def load_camera_gt_labels(label_path):
    boxes = []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            obj_type = parts[0]
            h, w, l = map(float, parts[8:11])
            x, y, z, = map(float, parts[11:14])
            rot_y = float(parts[14])
            class_idx = CLASS_MAP.get(obj_type, 0)
            boxes.append([x, y, z, h, w, l, rot_y, class_idx])

    return np.array(boxes, dtype=np.float32)

def load_camera_pred_labels(label_path, min_confidence=0.01):
    boxes, scores, labels = [], [], []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            obj_type = parts[0]
            h, w, l = map(float, parts[8:11]) 
            x, y, z = map(float, parts[11:14])
            rot_y = float(parts[14])
            confidence = float(parts[15])
            class_idx = CLASS_MAP.get(obj_type, 0)

            if confidence > min_confidence:
                boxes.append([x, y, z, h, w, l, rot_y, class_idx])
                scores.append(confidence)
                labels.append(class_idx)

    return np.array(boxes, dtype=np.float32), np.array(scores), np.array(labels)

def load_image(image_path, f_no=None):

    if f_no is not None:
        filename = f"{f_no:06d}.png"
        full_path = Path(image_path) / filename
    else:
        full_path = Path(image_path)

    img = cv2.imread(str(full_path))

    return img

def load_P2_matrix(pkl_path, lidar_idx):
    with open(pkl_path, 'rb') as f:
        val_data = pickle.load(f)

    for matrix_data in val_data:
        if matrix_data['point_cloud']['lidar_idx'] == lidar_idx:
            return matrix_data['calib']['P2']

def draw_projected_box3d(image, box3d_camera, P2, color=None, thickness=1, is_gt=False):
    x, y, z, h, w, l, ry, class_idx = box3d_camera

    x_corners = [l/2, l/2, -l/2, -l/2, l/2, l/2, -l/2, -l/2]
    y_corners = [0, 0, 0, 0, -h, -h, -h, -h]
    z_corners = [w/2, -w/2, -w/2, w/2, w/2, -w/2, -w/2, w/2]
    corners = np.vstack([x_corners, y_corners, z_corners])  # (3,8)

    R = np.array([
        [np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])
    corners_rot = R @ corners
    corners_trans = corners_rot + np.array([[x], [y], [z]])

    corners_hom = np.vstack([corners_trans, np.ones((1,8))])
    img_pts = P2 @ corners_hom
    img_pts = img_pts[:2] / img_pts[2]

    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0), 
        (4, 5), (5, 6), (6, 7), (7, 4), 
        (0, 4), (1, 5), (2, 6), (3, 7) 
    ]
    """
    # https://github.com/kuixu/kitti_object_vis/blob/12ce0a2348f6e1405c502bf32e51d76d3a970396/kitti_util.py#L669
      1 -------- 0
     /|         /|
    2 -------- 3 .
    | |        | |
    . 5 -------- 4
    |/         |/
    6 -------- 7
    """
    img = image.copy()

    if is_gt:
        color = (0, 0, 255)
    else:
        class_idx = int(class_idx)
        rgb = box_colormap[class_idx]
        color = (int(rgb[2]*255), int(rgb[1]*255), int(rgb[0]*255))

    for i, j in edges:
        pt1 = tuple(img_pts[:, i].astype(int))
        pt2 = tuple(img_pts[:, j].astype(int))
        cv2.line(img, pt1, pt2, color, thickness)
    
    if is_gt:
        # Draw heading diagonal
        corner_1 = tuple(img_pts[:, 1].astype(int))
        corner_4 = tuple(img_pts[:, 4].astype(int))
        cv2.line(img, corner_1, corner_4, (255, 255, 0), 2)
        
        corner_0 = tuple(img_pts[:, 0].astype(int))
        corner_5 = tuple(img_pts[:, 5].astype(int))
        cv2.line(img, corner_0, corner_5, (255, 255, 0), 2)

    return img

if __name__ == "__main__":
    
    base_dir = Path("/home/heizung1/view-of-delft-dataset/vod/label_transformation")
    gt_path = base_dir / "kitti_gt_annos_2/gt_lidar_to_camera_labels_2/000006.txt"
    pred_path = base_dir / "predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp32/000006.txt"
    img_path = base_dir / "validation_images"
    pkl_path = "validation_pickle/kitti_val_dataset.pkl"
    save_image = True

    frame_id = gt_path.stem
    frame_number = int(frame_id)

    camera_image = load_image(image_path=img_path, f_no=frame_number)
    P2_matrix = load_P2_matrix(pkl_path=pkl_path, lidar_idx=frame_id)
    gt_boxes = load_camera_gt_labels(label_path=gt_path)
    pred_boxes, pred_scores, pred_labels = load_camera_pred_labels(label_path=pred_path, min_confidence=0.01)

    for box in gt_boxes:
        camera_image = draw_projected_box3d(image=camera_image, box3d_camera=box, P2=P2_matrix, is_gt=True)
    
    for box in pred_boxes:
        camera_image = draw_projected_box3d(image=camera_image, box3d_camera=box, P2=P2_matrix, is_gt=False)

    cv2.imshow('Camera Predictions vs. GT', camera_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    if save_image:
        cv2.imwrite("images/cam_predictions_result.png", camera_image)